# 00 · Setup — environment check

Every module starts with `%run ../../setup/00_setup`. This notebook resolves the
participant catalog and exports shared variables.

Trainer automation can pass `TARGET_CATALOG` when running
`data/generate_training_dataset.ipynb` for another participant catalog.

In [ ]:
import re

CATALOG_PREFIX = "retailhub"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
DEFAULT_SCHEMA = "default"
VOLUME_NAME = "datasets"

def get_notebook_parameter(name):
    try:
        return dbutils.widgets.get(name).strip()
    except Exception:
        return ""

def resolve_user_slug(current_user):
    identity = current_user.lower()
    local_part = identity.split("@")[0]

    if "trainer" in identity or "trener" in identity or local_part == "krzysztof.burejza":
        return "trainer"

    user_slug = re.sub(r"[^a-zA-Z0-9]", "_", local_part).lower()
    return re.sub(r"_+", "_", user_slug).strip("_") or "user"

current_user = spark.sql("SELECT current_user()").first()[0]
TARGET_CATALOG = get_notebook_parameter("TARGET_CATALOG")

if TARGET_CATALOG:
    CATALOG = TARGET_CATALOG
    prefix = f"{CATALOG_PREFIX}_"
    user_slug = CATALOG[len(prefix):] if CATALOG.startswith(prefix) else CATALOG
else:
    user_slug = resolve_user_slug(current_user)
    CATALOG = f"{CATALOG_PREFIX}_{user_slug}"

BRONZE = f"{CATALOG}.{BRONZE_SCHEMA}"
SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
DATASET_PATH = f"/Volumes/{CATALOG}/{DEFAULT_SCHEMA}/{VOLUME_NAME}"

print("User:", current_user)
print("Catalog:", CATALOG)
if TARGET_CATALOG:
    print("Catalog override:", TARGET_CATALOG)

In [ ]:
catalogs = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}
if CATALOG not in catalogs:
    raise Exception(f"Catalog {CATALOG} not found. Ask the trainer to run setup/00_pre_config.")

spark.sql(f"USE CATALOG {CATALOG}")

schemas = {r[0] for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()}
missing = [s for s in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA] if s not in schemas]
if missing:
    raise Exception(f"Missing schemas in {CATALOG}: {missing}")

print("[OK] Environment ready")
print("BRONZE:", BRONZE)
print("SILVER:", SILVER)
print("GOLD:", GOLD)
print("DATASET_PATH:", DATASET_PATH)